### Installation

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Unsloth

In [2]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 15 trillion tokens model 2x faster!
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # We also uploaded 4bit for 405b!
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit", # New Mistral 12b 2x faster!
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",        # Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.1.1: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
from tqdm import tqdm
import torch
import time

# 1. Đọc file CSV và lấy 50 dòng đầu
# Thay đổi đường dẫn nếu file nằm trong thư mục khác trên Colab
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Alignment/JSON2_4653_5815.csv")
subset_data = df.head(50).copy() # Lấy 50 dòng đầu

# 2. Định nghĩa Prompt (Cấu trúc Alpaca cho dịch thuật)
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Translate the following text from Chinese to Vietnamese.

### Input:
{}

### Response:
{}"""

# 3. Hàm thực hiện dịch
def run_inference(chinese_text):
    inputs = tokenizer(
        [
            alpaca_prompt.format(
                chinese_text, # Input: Câu tiếng Trung
                "",           # Output: Để trống để mô hình tự điền
            )
        ], return_tensors = "pt").to("cuda")

    # Generate
    outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)
    decoded_output = tokenizer.batch_decode(outputs)[0]

    # Tách lấy phần Response
    response_start = "### Response:\n"
    response = decoded_output.split(response_start)[1].replace("<|end_of_text|>", "").strip()
    return response

# 4. Chạy vòng lặp dịch 50 câu
print("Đang thực hiện dịch 50 câu đầu...")
hypotheses = []

start_time = time.time()

for index, row in tqdm(subset_data.iterrows(), total=subset_data.shape[0]):
    src_text = row['src_lang'] # Cột tiếng Trung
    pred = run_inference(src_text)
    hypotheses.append(pred)


end_time = time.time()
total_time = end_time - start_time
avg_time = total_time / len(subset_data)

# Lưu kết quả vào cột mới
subset_data['hypothesis'] = hypotheses

# 5. Lưu file kết quả (theo yêu cầu PDF: dạng CSV)
output_filename = "Llama3_1_Survey_Result_50.csv"
subset_data.to_csv(output_filename, index=False)
print(f"Đã lưu kết quả khảo sát vào file: {output_filename}")

print(f"\n✅ Đã hoàn thành!")
print(f"Tổng thời gian chạy: {total_time:.2f} giây")
print(f"Tốc độ trung bình: {avg_time:.2f} giây/câu")

# Hiển thị vài dòng kết quả
print(subset_data[['src_lang', 'tgt_lang', 'hypothesis']].head())

Đang thực hiện dịch 50 câu đầu...


100%|██████████| 50/50 [02:57<00:00,  3.55s/it]

Đã lưu kết quả khảo sát vào file: Llama3_1_Survey_Result_50.csv

✅ Đã hoàn thành!
Tổng thời gian chạy: 177.75 giây
Tốc độ trung bình: 3.55 giây/câu
                                  src_lang  \
0                             我认为这将是主要的事情。   
1  我认为，同意某人所说，应该选择HSV-1血清阳性且携带APOE4等位基因的人。   
2                                 我非常想说这个。   
3                  史蒂文·雅各布森：但是马克，我可以提个建议吗？   
4                        我相信这是从许多问题中得出的结论。   

                                            tgt_lang  \
0                 Tôi nghĩ rằng đó sẽ là điều chính.   
1  Tôi nghĩ, đồng ý với người nào đó nói rằng nên...   
2                         Tôi rất muốn nói điều này.   
3  Steven Jacobson: Nhưng Mack, tôi có thể đưa ra...   
4  Tôi chắc chắn điều này được tìm kiếm từ rất nh...   

                                          hypothesis  
0  Tôi nghĩ rằng điều này sẽ là điều quan trọng n...  
1  Tôi nghĩ rằng, đồng ý với một người nói, nên c...  
2                         Tôi rất muốn nói điều này.  
3  史蒂文·雅各布森：但是马克，我

In [ ]:
!pip install sacrebleu

import sacrebleu

# Chuẩn bị dữ liệu
# refs: Danh sách các câu mẫu (Reference) - Sacrebleu yêu cầu dạng list of lists
refs = [subset_data['tgt_lang'].tolist()]
# sys: Danh sách các câu máy dịch (Hypothesis)
sys = subset_data['hypothesis'].tolist()

print("-" * 30)
print("KẾT QUẢ ĐÁNH GIÁ (SURVEY METRICS):")
print("-" * 30)

# 1. Tính BLEU (Chỉ số chính - Càng cao càng tốt)
bleu = sacrebleu.corpus_bleu(sys, refs)
print(f"BLEU score: {bleu.score:.2f}")

# 2. Tính ChrF (Độ khớp ký tự - Càng cao càng tốt)
chrf = sacrebleu.corpus_chrf(sys, refs)
print(f"ChrF score: {chrf.score:.2f}")

# 3. Tính TER (Tỷ lệ chỉnh sửa - CÀNG THẤP CÀNG TỐT)
ter = sacrebleu.corpus_ter(sys, refs)
print(f"TER score : {ter.score:.2f}")
print("-" * 30)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 6.7 MB/s eta 0:00:00
------------------------------
KẾT QUẢ ĐÁNH GIÁ (SURVEY METRICS):
------------------------------
BLEU score: 21.85
ChrF score: 41.38
TER score : 89.45
------------------------------
